# 🔴 Famalicão — Vulnerabilidade após Perda
**Projeto Final — Análise de Risco na 1ª Fase de Construção**

Este notebook calcula as métricas de vulnerabilidade defensiva após perdas de bola na construção do FC Famalicão, utilizando dados da API StatsBomb.

### Instruções
Corre as células **uma a uma**, de cima para baixo:
1. **Instalar dependências**
2. **Configuração e funções auxiliares**
3. **Buscar jogos disponíveis**
4. **Calcular métricas por jogo**
5. **Exportar CSV**


## 1. Instalar dependências

In [ ]:
# Instalar as bibliotecas necessárias
!pip install -q requests pandas


## 2. Configuração

Credenciais, parâmetros da competição e funções auxiliares.

| Parâmetro | Valor | Descrição |
|---|---|---|
| COMPETITION_ID | 13 | Liga Portugal |
| SEASON_ID | 318 | Época 2025/26 |
| TRANSITION_WINDOW | 10s | Janela de tempo após perda para analisar a transição adversária |
| DEEP_ENTRY_X | 80 | Limite do último terço no campo StatsBomb (0-120 em x) |


In [ ]:
import requests
import pandas as pd
from getpass import getpass

# ── Credenciais da API StatsBomb ──────────────────────────────────────────
# Pedir as credenciais ao utilizador no momento da execução,
# para não ficarem guardadas em texto no notebook.
USER = input("Introduz o utilizador da API StatsBomb: ").strip()
PASS = getpass("Introduz a password da API StatsBomb: ").strip()

# ── Parâmetros da competição ──────────────────────────────────────────────
COMPETITION_ID    = 13        # Liga Portugal
SEASON_ID         = 318       # Época 2025/26
TEAM_NAME         = "Famalicão"
TRANSITION_WINDOW = 10        # segundos após perda para analisar resposta adversária
DEEP_ENTRY_X      = 80        # coordenada x do limite do último terço (campo 0-120)
team_key          = "Famalic" # substring para identificar o Famalicão
auth              = (USER, PASS)

# ── Funções auxiliares ────────────────────────────────────────────────────

def get_json(url):
    """Faz pedido GET à API StatsBomb e devolve o JSON."""
    r = requests.get(url, auth=auth)
    r.raise_for_status()
    return r.json()

def ts_to_seconds(ts):
    """Converte timestamp 'HH:MM:SS.mmm' em segundos totais."""
    try:
        h, m, rest = str(ts).split(":")
        s, ms = rest.split(".")
        return int(h)*3600 + int(m)*60 + int(s) + int(ms)/1000
    except:
        return 0.0

def safe_x(loc):
    """Extrai a coordenada x de uma localização [x, y] ou devolve None."""
    if isinstance(loc, list) and len(loc) >= 1:
        return loc[0]
    return None

print("✅ Credenciais introduzidas e configuração carregada com sucesso")


## 3. Buscar jogos do Famalicão

Vai buscar todos os jogos **disponíveis** do Famalicão na Liga Portugal 25/26.


In [ ]:
# ── Buscar lista de jogos da época via API StatsBomb ─────────────────────
print("A ir buscar jogos...")

url = (f"https://data.statsbombservices.com/api/v6/competitions/"
       f"{COMPETITION_ID}/seasons/{SEASON_ID}/matches")
matches = pd.json_normalize(get_json(url), sep=".")

HOME_COL = "home_team.home_team_name"
AWAY_COL = "away_team.away_team_name"

# Filtrar apenas jogos disponíveis do Famalicão (casa ou fora)
mask = (
    (matches["match_status"] == "available") &
    (
        matches[HOME_COL].str.contains(team_key, case=False, na=False) |
        matches[AWAY_COL].str.contains(team_key, case=False, na=False)
    )
)
team_matches = matches[mask].copy()
print(f"→ {len(team_matches)} jogos encontrados")

def get_opponent(row):
    """Devolve o nome do adversário (equipa que não é o Famalicão)."""
    if team_key.lower() in row[HOME_COL].lower():
        return row[AWAY_COL]
    return row[HOME_COL]

team_matches["opponent"]   = team_matches.apply(get_opponent, axis=1)
team_matches["match_date"] = pd.to_datetime(team_matches["match_date"])
team_matches = team_matches.sort_values("match_date").reset_index(drop=True)

# Mostrar lista de jogos encontrados
team_matches[["match_id", "match_date", "opponent"]]


## 4. Calcular métricas de vulnerabilidade

Para cada jogo, o script:
1. **Identifica as perdas** do Famalicão no 1º terço (x ≤ 40): `Miscontrol`, `Dispossessed` ou `Pass` incompleto
2. **Analisa os 10 segundos seguintes** a cada perda — o que o adversário faz após recuperar a bola
3. **Calcula 4 métricas principais:**

| Métrica | Descrição |
|---|---|
| **xG sofrido após perda** | Soma do xG StatsBomb dos remates adversários nos 10s após cada perda |
| **Exposição Defensiva (%)** | % de perdas que resultaram em pelo menos um remate adversário |
| **Counterpress Efficiency (%)** | % de perdas em que o Famalicão recuperou a bola em menos de 5s |
| **Transition Risk Index (TRI)** | Índice composto 0-100: xG×40% + remates/perda×25% + entradas últ.terço×20% + progressão×15% |

O **VAP** (índice original) é também calculado para referência histórica.

> ⚠️ Esta célula pode demorar **2-3 minutos** — faz um pedido à API por cada jogo.


In [ ]:
results = []

for i, row in team_matches.iterrows():
    match_id   = int(row["match_id"])
    match_date = row["match_date"].strftime("%Y-%m-%d")
    opponent   = row["opponent"]
    print(f"[{i+1}/{len(team_matches)}] {match_date} vs {opponent} ...", end=" ")

    # ── Carregar todos os eventos do jogo (API StatsBomb v8) ──────────────
    ev = pd.json_normalize(
        get_json(f"https://data.statsbombservices.com/api/v8/events/{match_id}"),
        sep="."
    )

    # Normalizar colunas que chegam como dicionários {id, name} → só o nome
    for col in ["type.name", "play_pattern.name", "pass.outcome.name", "possession_team.name"]:
        if col in ev.columns:
            ev[col] = ev[col].apply(lambda x: x.get("name") if isinstance(x, dict) else x)

    # Adicionar tempo em segundos e coordenada x de cada evento
    ev["time_s"] = ev["timestamp"].apply(ts_to_seconds) + (ev["period"] - 1) * 45 * 60
    ev["loc_x"]  = ev["location"].apply(safe_x)

    # Nomes das colunas que vamos usar
    TYPE_COL    = "type.name"
    POSS_COL    = "possession_team.name"
    PATTERN_COL = "play_pattern.name"
    PASS_OUT    = "pass.outcome.name"
    SHOT_XG     = "shot.statsbomb_xg"  # xG calculado pelo modelo da StatsBomb

    # ── Filtrar apenas eventos do Famalicão ───────────────────────────────
    fama_ev = ev[ev[POSS_COL].apply(lambda x: team_key.lower() in str(x).lower())].copy()

    # ── Identificar perdas na 1ª fase de construção (x ≤ 40) ─────────────
    # Tipos de perda considerados:
    #   - Miscontrol: má receção / mau controlo da bola
    #   - Dispossessed: desarme pelo adversário
    #   - Pass Incomplete: passe falhado no 1º terço
    loss_mask = (
        (fama_ev[TYPE_COL].isin({"Miscontrol", "Dispossessed"})) |
        (
            (fama_ev[TYPE_COL] == "Pass") &
            (fama_ev.get(PASS_OUT, pd.Series(dtype=str)) == "Incomplete") &
            (fama_ev["loc_x"].apply(lambda x: x <= 40 if x is not None else False))
        )
    )
    losses = fama_ev[loss_mask].copy()
    n_losses = len(losses)  # total de perdas identificadas neste jogo

    # ── Inicializar contadores ────────────────────────────────────────────
    hp = ca = cl = dp = 0   # contadores para o VAP (referência histórica)
    xg_total         = 0.0  # xG total sofrido após perdas
    shots_after_loss = 0    # total de remates sofridos após perdas
    losses_leading_to_shot = 0   # perdas que resultaram em remate (→ Exposição Defensiva)
    opp_progression_total  = 0.0 # metros totais de progressão adversária após perdas

    # ── Analisar a transição adversária após cada perda ───────────────────
    for _, le in losses.iterrows():
        loss_time       = le["time_s"]       # momento da perda (em segundos)
        loss_possession = le["possession"]   # número da posse em que ocorreu a perda
        loss_x          = le["loc_x"] or 40  # posição x da perda no campo

        # Filtrar eventos adversários na janela de 10s após a perda
        opp_after = ev[
            (ev["possession"] > loss_possession) &             # posse seguinte (adversário já tem a bola)
            (ev["time_s"] >= loss_time) &                      # a partir do momento da perda
            (ev["time_s"] <= loss_time + TRANSITION_WINDOW) &  # dentro dos 10 segundos
            (~ev[POSS_COL].apply(lambda x: team_key.lower() in str(x).lower())) # não é o Famalicão
        ]
        if opp_after.empty:
            continue

        # Verificar se houve remate nesta transição
        shots = opp_after[opp_after[TYPE_COL] == "Shot"]
        if len(shots) > 0:
            losses_leading_to_shot += 1  # esta perda gerou remate → conta para Exposição Defensiva

        # Analisar cada remate adversário
        for _, shot in shots.iterrows():
            pp = str(shot.get(PATTERN_COL, ""))

            # Classificar o remate (para o VAP histórico)
            if "Counter" in pp:
                ca += 1   # remate em contra-ataque (play_pattern = "From Counter")
            elif shot["time_s"] - loss_time <= 5:
                hp += 1   # remate em pressão imediata (menos de 5s após a perda)

            # xG do remate — valor calculado pelo modelo StatsBomb (já vem nos dados)
            xg_val = shot.get(SHOT_XG, 0) or 0
            if xg_val >= 0.15:
                cl += 1   # situação clara de golo (xG alto)
            xg_total += xg_val  # acumula o xG total do jogo
            shots_after_loss += 1

        # Contar entradas no último terço adversário (x ≥ 80)
        dp += len(opp_after[opp_after["loc_x"].apply(
            lambda x: x >= DEEP_ENTRY_X if x is not None else False)])

        # Calcular progressão adversária após a perda (em metros)
        # No terço defensivo do Famalicão, o adversário avança quando x DECRESCE
        opp_x_vals = opp_after["loc_x"].dropna()
        if len(opp_x_vals) > 0:
            progression = loss_x - opp_x_vals.min()  # metros avançados em direção à baliza
            opp_progression_total += max(0, progression)

    # ── Counterpress Efficiency ───────────────────────────────────────────
    # % de perdas em que o Famalicão recuperou a bola em menos de 5 segundos
    # Usa o campo 'counterpress' da StatsBomb (marcado em recuperações imediatas após perda)
    cp_recoveries = 0
    if n_losses > 0:
        for _, le in losses.iterrows():
            loss_time       = le["time_s"]
            loss_possession = le["possession"]

            # Procurar recuperações do Famalicão nos 5s após a perda
            recovery_window = ev[
                (ev["possession"] >= loss_possession) &
                (ev["time_s"] >= loss_time) &
                (ev["time_s"] <= loss_time + 5) &
                (ev[POSS_COL].apply(lambda x: team_key.lower() in str(x).lower())) &
                (ev[TYPE_COL].isin({"Ball Recovery", "Interception", "Duel"}))
            ]
            # Filtrar só eventos marcados como counterpress pela StatsBomb
            if "counterpress" in ev.columns:
                recovery_window = recovery_window[recovery_window["counterpress"] == True]
            if len(recovery_window) > 0:
                cp_recoveries += 1

    # ── Calcular métricas finais ──────────────────────────────────────────

    # Exposição Defensiva: % de perdas que geraram pelo menos um remate adversário
    exposicao_defensiva = round(losses_leading_to_shot / n_losses * 100, 1) if n_losses > 0 else 0

    # Counterpress Efficiency: % de perdas com recuperação imediata em <5s
    counterpress_eff = round(cp_recoveries / n_losses * 100, 1) if n_losses > 0 else 0

    # Progressão adversária média por perda (metros)
    avg_opp_progression = round(opp_progression_total / n_losses, 2) if n_losses > 0 else 0

    # ── Transition Risk Index (TRI) — índice composto normalizado 0-100 ───
    # Combina as 4 métricas com pesos definidos, cada uma normalizada pelo seu valor máximo de referência
    shots_p_loss = shots_after_loss / n_losses if n_losses > 0 else 0
    dp_p_loss    = dp / n_losses if n_losses > 0 else 0
    prog_norm    = min(avg_opp_progression / 40, 1)  # normalizado: 40m = máximo de referência

    tri = round(
        0.40 * min(xg_total / 2, 1) * 100 +          # xG sofrido        (peso 40%, ref. max = 2.0 xG)
        0.25 * min(shots_p_loss / 0.5, 1) * 100 +    # remates / perda   (peso 25%, ref. max = 0.5)
        0.20 * min(dp_p_loss / 5, 1) * 100 +          # entradas últ. terço / perda (peso 20%, ref. max = 5)
        0.15 * prog_norm * 100,                        # progressão norm.  (peso 15%)
    1)

    # VAP — índice original para referência histórica
    # Fórmula: 4×remates pressão + 3×remates contra-ataque + 2×remates claros + 0.1×entradas últ.terço
    vap = 4*hp + 3*ca + 2*cl + 0.1*dp

    print(f"perdas={n_losses} | xG={xg_total:.2f} | exp={exposicao_defensiva}% | cp={counterpress_eff}% | TRI={tri}")

    results.append({
        "match_id":    match_id,
        "match_date":  match_date,
        "opponent":    opponent,
        "team_name":   TEAM_NAME,
        # ── Contadores base ────────────────────────────────────────────────
        "n_losses":                                    n_losses,
        "team_match_high_press_shots_conceded":        hp,
        "team_match_counter_attacking_shots_conceded": ca,
        "team_match_shots_in_clear_conceded":          cl,
        "team_match_deep_progressions_conceded":       dp,
        # ── Métricas principais ────────────────────────────────────────────
        "team_match_xg_conceded_after_loss":           round(xg_total, 3),
        "exposicao_defensiva_pct":                     exposicao_defensiva,
        "counterpress_efficiency_pct":                 counterpress_eff,
        "avg_opp_progression_m":                       avg_opp_progression,
        "transition_risk_index":                       tri,
        # ── VAP (referência histórica) ─────────────────────────────────────
        "VAP":                                         round(vap, 2),
    })

print(f"\n✅ Cálculo concluído para {len(results)} jogos")


## 5. Exportar CSV

Gera o ficheiro `vulnerabilidade_perda_real_novasmetricas.csv` com todas as métricas calculadas.
Este ficheiro alimenta o dashboard Streamlit — faz o upload no GitHub após descarregar.


In [ ]:
# ── Criar DataFrame e exportar para CSV ──────────────────────────────────
df_out = pd.DataFrame(results)

# Guardar com separador ";" (compatível com o dashboard Streamlit)
df_out.to_csv("vulnerabilidade_perda_real_novasmetricas.csv", sep=";", index=False, encoding="utf-8")

print(f"✅ Ficheiro exportado: vulnerabilidade_perda_real_novasmetricas.csv")
print(f"   Jogos: {len(df_out)}")
print(f"   Colunas: {list(df_out.columns)}")
print()

# Mostrar resumo das métricas principais por jogo
display(df_out[[
    "match_date",
    "opponent",
    "n_losses",
    "team_match_xg_conceded_after_loss",
    "exposicao_defensiva_pct",
    "counterpress_efficiency_pct",
    "transition_risk_index",
    "VAP"
]].round(2))
